# SoundScape AI | Computer Vision — Final Production Training

**Vision-Guided Controllable Nature Sound Generation**  
**Release:** CV v1.0 • 14 November 2026 • Google Colab (GPU recommended)

### What this final notebook delivers

1. Source-data audit, EDA, exact-duplicate removal, and leakage-controlled train/validation/test splits.
2. Full-dataset **ImageNet-pretrained MobileNetV3-Small**, classifier training, **partial backbone fine-tuning**, and validation-based model selection.
3. Evaluation on the untouched test set, per-class metrics, confusion matrix, learning curves, and confidence calibration.
4. Validation-calibrated **uncertain-scene abstention** for integration; editable user controls for audio characteristics.
5. Real-image upload testing, production checkpoint export (`scene_mobilenetv3.pt`), reproducibility and test reports, downloadable release ZIP.
6. Stable JSON contract for the SoundScape FastAPI app, ready to connect to the audio teammate's model.

> **Model scope:** six verified visual labels — `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`.  
> It does **not** learn `desert`, `river`, `lake`, `waterfall`, `rain` or `wind` from this dataset. Those require separately labeled images / independent evaluation. Uploaded photos cannot prove what was actually audible. Unsupported scenes remain manual choices or *unvalidated* CLIP suggestions.

**Execution:** `Runtime → Change runtime type → T4 GPU`, then **Run all**. The notebook saves outputs under `/content/soundscape_cv_final/` (or the current working directory outside Colab). Network is needed to download the public dataset and pretrained weights. **No training/evaluation results are assumed in advance.**

## 0. Runtime & dependencies

Use a **fresh Colab runtime** for the released notebook. Install the four data/scientific packages if required. Do not upgrade the preinstalled `torch` / `torchvision` pair independently. If Colab requires a restart after installation, restart once and run from the beginning.

In [ ]:
%pip -q install "datasets>=3.0,<5" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4" "pillow>=10,<13" "scipy>=1.12,<2"

In [ ]:
import os, sys, platform, random, json, time, hashlib, copy, shutil, warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, ConfusionMatrixDisplay, confusion_matrix)
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp, softmax

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_ID = 'prithivMLmods/OpenScene-Classification'
EXPECTED_CLASSES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
OUT_DIR = Path('soundscape_cv_final'); OUT_DIR.mkdir(parents=True, exist_ok=True)

# FINAL full-dataset run: NO 300-per-class pilot cap.
HEAD_EPOCHS = 8
FINETUNE_EPOCHS = 10
PATIENCE = 3
BATCH_SIZE = 64 if DEVICE.type == 'cuda' else 16
NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0
TARGET_ACCEPTED_ACCURACY = 0.90
MIN_COVERAGE = 0.20

VERSIONS = {
    'python': sys.version.split()[0], 'torch': torch.__version__,
    'torchvision': __import__('torchvision').__version__,
    'datasets': __import__('datasets').__version__,
    'numpy': np.__version__, 'pillow': Image.__version__,
    'device': str(DEVICE), 'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
}
print(json.dumps(VERSIONS, indent=2, ensure_ascii=False))
if DEVICE.type != 'cuda':
    print('NOTE: CPU is supported but the full-dataset training will be slower.')

## 1. Load the verified visual dataset

The source has **16,345 labeled images, six classes, and one published train split**. This notebook creates its own stratified validation and untouched test subsets. The class distribution is read from the dataset, not hard-coded.

**Source:** https://huggingface.co/datasets/prithivMLmods/OpenScene-Classification  
**Licensing:** the dataset card says Apache-2.0, but underlying image provenance is not fully documented. Review before commercial deployment.

In [ ]:
raw = load_dataset(DATASET_ID, split='train')
assert 'image' in raw.column_names and 'label' in raw.column_names, raw.column_names
label_names = list(raw.features['label'].names)
assert label_names == EXPECTED_CLASSES, f'Dataset taxonomy changed: {label_names}'
labels_all = np.asarray(raw['label'], dtype=np.int64)
assert len(labels_all) == len(raw)
source_counts = Counter(label_names[int(y)] for y in labels_all)
print('Images in source:', len(raw))
print('Class distribution:', dict(source_counts))
print('Columns:', raw.column_names)

## 2. EDA and visual data-quality checks

Inspect class balance, representative images, resolutions, unreadable files, exact duplicate pixels, and likely perceptual duplicate *candidates*. Near-duplicate warnings are only diagnostic; the notebook does **not** silently delete images based on an approximate perceptual hash.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.bar(label_names, [source_counts[n] for n in label_names])
ax.set(title='Source dataset — images per class', ylabel='Image count')
plt.tight_layout(); plt.show()

fig, axs = plt.subplots(2, 3, figsize=(11, 7))
for ax, label in zip(axs.flat, label_names):
    idx = int(np.flatnonzero(labels_all == label_names.index(label))[0])
    ax.imshow(ImageOps.exif_transpose(raw[idx]['image']).convert('RGB'))
    ax.set_title(label); ax.axis('off')
plt.suptitle('Representative labeled examples');plt.tight_layout();plt.show()

In [ ]:
def exact_image_hash(im):
    im = ImageOps.exif_transpose(im).convert('RGB')
    size_bytes = f'{im.size[0]}x{im.size[1]}:'.encode()
    return hashlib.sha256(size_bytes + im.tobytes()).hexdigest()

def perceptual_dhash(im):
    # Reporting-only approximate fingerprint; it is NOT a proof of duplication.
    grey = ImageOps.exif_transpose(im).convert('L').resize((9,8))
    arr = np.asarray(grey, dtype=np.int16)
    return ''.join('1' if v else '0' for v in (arr[:,1:] > arr[:,:-1]).ravel())

seen_hashes, near_buckets = {}, defaultdict(list)
unique_indices, exact_duplicate_indices, invalid_indices = [], [], []
sizes = Counter()
for i in range(len(raw)):
    try:
        image = raw[i]['image']
        with_rgb = ImageOps.exif_transpose(image).convert('RGB')
        sizes[with_rgb.size] += 1
        digest = exact_image_hash(with_rgb)
        if digest in seen_hashes:
            exact_duplicate_indices.append(i)
            continue
        seen_hashes[digest] = i
        unique_indices.append(i)
        near_buckets[perceptual_dhash(with_rgb)].append(i)
    except (OSError, ValueError, TypeError) as exc:
        invalid_indices.append(i)

near_collision_groups = [ids for ids in near_buckets.values() if len(ids) > 1]
print('Readable, unique images:', len(unique_indices))
print('Removed exact-duplicate images:', len(exact_duplicate_indices))
print('Unreadable/invalid images:', len(invalid_indices))
print('Matching perceptual-hash groups (REVIEW ONLY):', len(near_collision_groups))
print('Most common resolutions:', sizes.most_common(6))
assert len(unique_indices) > 100, 'Too few usable training images.'

eda_report = {
    'source_rows': len(raw), 'usable_unique_rows': len(unique_indices),
    'exact_duplicates_removed': len(exact_duplicate_indices),
    'invalid_images_excluded': len(invalid_indices),
    'perceptual_hash_collision_groups_review_only': len(near_collision_groups),
    'class_counts_source': dict(source_counts),
    'frequent_image_sizes': [{'size':list(size),'count':count} for size,count in sizes.most_common(10)],
}
(OUT_DIR/'eda_report.json').write_text(json.dumps(eda_report,indent=2),encoding='utf-8')

## 3. Leakage-controlled splits (70% / 15% / 15%)

Exact duplicate images are removed *before* splitting. The test split is never used for model choice, calibration, or threshold selection. The same seed and indices are exported to reproduce the experiment. Approximate near-duplicates may still exist and are reported as a limitation.

In [ ]:
unique_indices = np.asarray(unique_indices,dtype=np.int64)
unique_labels = labels_all[unique_indices]
train_idx, temp_idx = train_test_split(
    unique_indices, test_size=0.30, stratify=unique_labels, random_state=SEED)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=labels_all[temp_idx], random_state=SEED)
assert not(set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx) or set(val_idx) & set(test_idx))
assert len(train_idx)+len(val_idx)+len(test_idx) == len(unique_indices)
for split_name, indices in [('train',train_idx),('validation',val_idx),('test',test_idx)]:
    print(split_name, len(indices), Counter(label_names[int(y)] for y in labels_all[indices]))
np.savez_compressed(OUT_DIR/'split_indices.npz',train=train_idx,validation=val_idx,test=test_idx)

## 4. Training transforms and data loaders

Training augmentations are applied **only** to the training images. Validation/test and app inference use the exact same deterministic `weights.transforms()` preprocessing.

In [ ]:
weights = MobileNet_V3_Small_Weights.DEFAULT
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.12),
    transforms.ToTensor(),
    transforms.Normalize(mean=(.485,.456,.406),std=(.229,.224,.225)),
])
eval_transform = weights.transforms()

class SceneDataset(Dataset):
    def __init__(self, data, indices, transform):
        self.data, self.indices, self.transform = data, [int(i) for i in indices], transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, position):
        item = self.data[self.indices[position]]
        image = ImageOps.exif_transpose(item['image']).convert('RGB')
        return self.transform(image), int(item['label'])

train_loader = DataLoader(SceneDataset(raw,train_idx,train_transform),batch_size=BATCH_SIZE,
                          shuffle=True,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
val_loader = DataLoader(SceneDataset(raw,val_idx,eval_transform),batch_size=BATCH_SIZE,
                        shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
test_loader = DataLoader(SceneDataset(raw,test_idx,eval_transform),batch_size=BATCH_SIZE,
                         shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'))
print('Data loaders created:',len(train_loader),len(val_loader),len(test_loader),'batches')

## 5. Architecture, training helpers, frozen BatchNorm handling

**Two real stages:**

- **Stage A:** freeze visual features, train MobileNetV3 classifier (head training baseline).
- **Stage B:** unfreeze the last **three** feature blocks and fine-tune them with a smaller learning rate.

Prevent BatchNorm running-statistics drift in frozen blocks; this is a common hidden source of apparent improvements when the backbone is supposedly frozen. Checkpoints are selected by validation macro-F1 with early stopping and later compared **only on validation**.

In [ ]:
def build_model(pretrained=True):
    m = mobilenet_v3_small(weights=weights if pretrained else None)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features,len(label_names))
    return m

def state_on_cpu(m):
    return {key: tensor.detach().cpu().clone() for key,tensor in m.state_dict().items()}

def stage_trainability(m, phase):
    for p in m.features.parameters():p.requires_grad = False
    for p in m.classifier.parameters():p.requires_grad = True
    if phase == 'finetune':
        for p in m.features[-3:].parameters():p.requires_grad=True
    elif phase != 'head':
        raise ValueError(phase)

def keep_frozen_bn_stable(m):
    # Called after model.train(); stop running-mean/var updates in frozen feature blocks.
    for mod in m.features.modules():
        if isinstance(mod, nn.BatchNorm2d):
            if not any(p.requires_grad for p in mod.parameters()):
                mod.eval()

@torch.inference_mode()
def collect_outputs(m, loader):
    m.eval(); logits_list=[]; truth=[]; total_loss=[]
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True)
        logits=m(x).float().cpu()
        logits_list.append(logits)
        truth.append(y.cpu())
        total_loss.append(nn.functional.cross_entropy(logits,y.cpu(),reduction='sum').item())
    all_logits=torch.cat(logits_list).numpy()
    all_y=torch.cat(truth).numpy()
    return all_logits,all_y,float(sum(total_loss)/len(all_y))

def summary_metrics(logits, y, loss=None):
    pred=np.argmax(logits,axis=1)
    return {'loss': None if loss is None else float(loss),
            'accuracy':float(accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0))}

def evaluate(m,loader):
    logits,y,loss=collect_outputs(m,loader)
    return summary_metrics(logits,y,loss),logits,y

model=build_model(pretrained=True).to(DEVICE)
stage_trainability(model,'head')
print('Total params:',sum(p.numel() for p in model.parameters()))
print('Trainable head-stage params:',sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def train_stage(model, phase, train_loader, val_loader, epochs, lr, patience):
    stage_trainability(model,phase)
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=lr,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,epochs),eta_min=lr*0.1)
    # torch.amp signature varies by version; autocast uses torch.autocast.
    scaler=torch.amp.GradScaler('cuda',enabled=(DEVICE.type=='cuda'))
    best_f1=-1.; best_state=None; best_epoch=0; stale=0; history=[]
    for epoch in range(1,epochs+1):
        start=time.monotonic()
        model.train();keep_frozen_bn_stable(model)
        running_loss=0.;seen=0
        for x,y in train_loader:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=(DEVICE.type=='cuda')):
                logits=model(x)
                loss=nn.functional.cross_entropy(logits,y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(params,1.0)
            scaler.step(optimizer);scaler.update()
            running_loss+=float(loss.item())*len(y);seen+=len(y)
        metrics,_,_=evaluate(model,val_loader)
        row={'stage':phase,'epoch':epoch,'train_loss':running_loss/seen,
             'val_loss':metrics['loss'],'val_accuracy':metrics['accuracy'],
             'val_macro_f1':metrics['macro_f1'],'lr':optimizer.param_groups[0]['lr'],
             'seconds':round(time.monotonic()-start,1)}
        history.append(row)
        print(f"{phase} {epoch:02d}/{epochs} | train loss {row['train_loss']:.4f} "
              f"| val loss {row['val_loss']:.4f} | acc {row['val_accuracy']:.3f} "
              f"| macro-F1 {row['val_macro_f1']:.3f} | {row['seconds']:.1f}s")
        if metrics['macro_f1'] > best_f1 + 1e-5:
            best_f1=metrics['macro_f1'];best_state=state_on_cpu(model)
            best_epoch=epoch;stale=0
        else:
            stale+=1
        scheduler.step()
        if stale>=patience:
            print('Early stopping after',stale,'non-improving epochs')
            break
    assert best_state is not None
    model.load_state_dict(best_state)
    return pd.DataFrame(history),best_state,{'best_epoch':best_epoch,'best_val_macro_f1':best_f1}

## 6. Stage A — train the classification head

This stage trains **real parameters** in the classifier head while the image feature extractor stays frozen. It establishes a fair baseline for later backbone fine-tuning.

In [ ]:
head_history, head_state, head_meta = train_stage(
    model,'head',train_loader,val_loader,HEAD_EPOCHS,lr=8e-4,patience=PATIENCE)
head_metrics,head_val_logits,head_val_y = evaluate(model,val_loader)
print('HEAD-ONLY VALIDATION:',head_metrics)

## 7. Stage B — fine-tune the last visual blocks

We continue from the best head-trained checkpoint, unfreeze the last three feature blocks, and update the visual representation using a smaller learning rate. This is **actual transfer learning / fine-tuning** (not inference or prompt engineering).

In [ ]:
model.load_state_dict(head_state)
fine_history, fine_state, fine_meta = train_stage(
    model,'finetune',train_loader,val_loader,FINETUNE_EPOCHS,lr=5e-5,patience=PATIENCE)
fine_metrics,fine_val_logits,fine_val_y = evaluate(model,val_loader)
assert np.array_equal(head_val_y,fine_val_y)
print('FINETUNED VALIDATION:',fine_metrics)
print('Validation macro-F1 change:',round(fine_metrics['macro_f1']-head_metrics['macro_f1'],4))

## 8. Training curves and validation model selection

Selection uses **validation macro-F1 only**. If fine-tuning did not help, the better head-only checkpoint is exported instead of pretending the fine-tuned one won. The untouched test split is still not used.

In [ ]:
fig, axs=plt.subplots(1,2,figsize=(12,4))
for hist,label in [(head_history,'Head'),(fine_history,'Fine-tune')]:
    axs[0].plot(hist['epoch'],hist['train_loss'],marker='o',label=f'{label} train')
    axs[0].plot(hist['epoch'],hist['val_loss'],marker='x',label=f'{label} val')
    axs[1].plot(hist['epoch'],hist['val_macro_f1'],marker='o',label=label)
axs[0].set(title='Learning curves',xlabel='Epoch (per stage)',ylabel='Cross-entropy')
axs[1].set(title='Validation macro-F1',xlabel='Epoch (per stage)',ylabel='Macro-F1')
for ax in axs:ax.grid(alpha=.2);ax.legend()
plt.tight_layout();plt.savefig(OUT_DIR/'learning_curves.png',dpi=160);plt.show()

selected_phase='finetune' if fine_metrics['macro_f1']>head_metrics['macro_f1'] else 'head'
selected_state=fine_state if selected_phase=='finetune' else head_state
selected_val_logits=fine_val_logits if selected_phase=='finetune' else head_val_logits
model.load_state_dict(selected_state);model.eval()
print('Selected phase:',selected_phase)

## 9. Calibrate confidence on validation, then decide when to ask the user

Softmax outputs are **not** a validated probability of scene truth or proof that the input is from the training distribution. Temperature scaling and a selective-accuracy threshold can reduce overconfident *in-domain* errors. **They do not guarantee out-of-distribution detection.** The studio always allows users to edit the scene.

In [ ]:
def fit_temperature(logits,y):
    logits=np.asarray(logits,dtype=np.float64);y=np.asarray(y,dtype=np.int64)
    def nll(log_t):
        scaled=logits/np.exp(log_t)
        return float(np.mean(logsumexp(scaled,axis=1)-scaled[np.arange(len(y)),y]))
    result=minimize_scalar(nll,bounds=(-2.3,2.3),method='bounded',options={'xatol':1e-4})
    temperature=float(np.exp(result.x))
    return temperature,{'nll_before':nll(0.0),'nll_after':nll(result.x)}

def expected_calibration_error(probabilities,y,bins=12):
    conf=probabilities.max(axis=1);pred=probabilities.argmax(axis=1)
    correct=(pred==y)
    edges=np.linspace(0,1,bins+1);ece=0.
    for i in range(bins):
        idx=(conf>=edges[i]) & (conf<edges[i+1] if i<bins-1 else conf<=edges[i+1])
        if idx.any():ece+=float(idx.mean()*abs(correct[idx].mean()-conf[idx].mean()))
    return float(ece)

def choose_accept_threshold(probs,y,target=TARGET_ACCEPTED_ACCURACY,min_coverage=MIN_COVERAGE):
    confidence=probs.max(axis=1);pred=probs.argmax(axis=1)
    candidates=[]
    for threshold in np.linspace(.0,.995,200):
        keep=confidence>=threshold
        if keep.any():
            candidates.append({'threshold':float(threshold),'coverage':float(keep.mean()),
              'accepted_accuracy':float((pred[keep]==y[keep]).mean()),'accepted_count':int(keep.sum())})
    feasible=[r for r in candidates if r['accepted_accuracy']>=target and r['coverage']>=min_coverage]
    if feasible:
        selected=max(feasible,key=lambda r:r['coverage'])
        selected['meets_target_on_validation']=True
    else:
        # Conservative fallback. Explicitly does NOT claim the precision target was reached.
        selected=min(candidates,key=lambda r:abs(r['threshold']-max(.75,float(np.quantile(confidence,.75)))))
        selected['meets_target_on_validation']=False
    return selected,pd.DataFrame(candidates)

TEMPERATURE,temp_stats=fit_temperature(selected_val_logits,head_val_y)
val_probs=softmax(selected_val_logits/TEMPERATURE,axis=1)
accept_result,selective_curve=choose_accept_threshold(val_probs,head_val_y)
ACCEPT_THRESHOLD=float(accept_result['threshold'])
print('Calibration temperature:',round(TEMPERATURE,4))
print('Validation calibration:',temp_stats)
print('Expected calibration error:',round(expected_calibration_error(val_probs,head_val_y),4))
print('Selected abstention policy (validation):',accept_result)
selective_curve.to_csv(OUT_DIR/'validation_selective_accuracy.csv',index=False)

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].plot(selective_curve['threshold'],selective_curve['accepted_accuracy'])
axs[0].axhline(TARGET_ACCEPTED_ACCURACY,ls='--',color='grey',label='Requested validation target')
axs[0].axvline(ACCEPT_THRESHOLD,ls=':',color='black',label='Selected')
axs[0].set(xlabel='Minimum calibrated confidence',ylabel='Accuracy of accepted predictions',title='Selective accuracy (validation)')
axs[0].legend()
axs[1].plot(selective_curve['threshold'],selective_curve['coverage'])
axs[1].axvline(ACCEPT_THRESHOLD,ls=':',color='black')
axs[1].set(xlabel='Minimum calibrated confidence',ylabel='Fraction of images accepted',title='Coverage (validation)')
plt.tight_layout();plt.savefig(OUT_DIR/'calibration_policy.png',dpi=160);plt.show()

## 10. Final evaluation on the untouched test set

**Run once.** The chosen weights, temperature and accept threshold are already fixed using validation only. No adjustments to those choices should be made based on these results. Test accuracy and macro-F1 are **real outputs of this run**, not numbers copied from the earlier pilot.

In [ ]:
test_metrics,test_logits,test_y=evaluate(model,test_loader)
test_pred=np.argmax(test_logits,axis=1)
test_prob=softmax(test_logits/TEMPERATURE,axis=1)
keep=test_prob.max(axis=1)>=ACCEPT_THRESHOLD
selective_test={
    'coverage':float(keep.mean()),
    'accepted_count':int(keep.sum()),
    'accepted_accuracy':float((test_pred[keep]==test_y[keep]).mean()) if keep.any() else None,
    'expected_calibration_error':expected_calibration_error(test_prob,test_y),
}
print('HELD-OUT TEST:',json.dumps(test_metrics,indent=2))
print('SELECTIVE TEST:',json.dumps(selective_test,indent=2))
print(classification_report(test_y,test_pred,labels=list(range(len(label_names))),
                            target_names=label_names,zero_division=0,digits=3))
class_report=classification_report(test_y,test_pred,labels=list(range(len(label_names))),
                                   target_names=label_names,zero_division=0,output_dict=True)
pd.DataFrame(class_report).T.to_csv(OUT_DIR/'per_class_test_metrics.csv')
errors_table=pd.DataFrame({'dataset_row':test_idx.astype(int),'true_label':[label_names[int(k)] for k in test_y],
    'predicted_label':[label_names[int(k)] for k in test_pred],
    'calibrated_score':test_prob.max(axis=1).astype(float),
    'correct':(test_y==test_pred).astype(bool)})
errors_table.to_csv(OUT_DIR/'held_out_test_predictions.csv',index=False)
fig,ax=plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay.from_predictions(test_y,test_pred,labels=list(range(len(label_names))),
    display_labels=label_names,cmap='Blues',xticks_rotation=30,ax=ax,colorbar=False)
ax.set_title('Held-out test: confusion matrix')
plt.tight_layout();plt.savefig(OUT_DIR/'confusion_matrix_test.png',dpi=160);plt.show()

## 11. Error analysis: inspect a sample of test failures

Here we inspect *failed predictions*, not adjust the training parameters based on the test set. It is important to understand confusion between glacier and mountain, sea and buildings, etc. Record errors for future **new-data collection**, not retroactive test optimization.

In [ ]:
wrong_positions=np.flatnonzero(test_pred != test_y)
print('Incorrect predictions:',len(wrong_positions),'out of',len(test_y))
if len(wrong_positions):
    rng=np.random.default_rng(SEED)
    selected_errors=rng.choice(wrong_positions,size=min(12,len(wrong_positions)),replace=False)
    cols=4;rows=int(np.ceil(len(selected_errors)/cols))
    fig,axs=plt.subplots(rows,cols,figsize=(14,rows*3.2))
    axs=np.atleast_1d(axs).ravel()
    for ax,pos in zip(axs,selected_errors):
        original_idx=int(test_idx[pos])
        ax.imshow(ImageOps.exif_transpose(raw[original_idx]['image']).convert('RGB'))
        ax.set_title(f'True: {label_names[test_y[pos]]}\nPred: {label_names[test_pred[pos]]}',fontsize=9)
        ax.axis('off')
    for ax in axs[len(selected_errors):]:ax.axis('off')
    plt.tight_layout();plt.savefig(OUT_DIR/'test_error_examples.png',dpi=150);plt.show()

## 12. Export real weights + complete scientific report

This checkpoint format is **compatible with the FastAPI app's `models/scene_mobilenetv3.pt`**. The app uses the checkpoint's temperature and acceptance threshold if available; older checkpoints without them still load with default values. The final release folder contains all evidence needed to reproduce and audit the result.

In [ ]:
NATURAL_SCENE_MAP={'buildings':'unknown','forest':'forest','glacier':'mountain',
                   'mountain':'mountain','sea':'coast','street':'unknown'}
CHECKPOINT=OUT_DIR/'scene_mobilenetv3.pt'
export_checkpoint={
    'state_dict':state_on_cpu(model),
    'classes':label_names,
    'training_phase':selected_phase,
    'dataset_id':DATASET_ID,
    'seed':SEED,
    'trained_samples':len(train_idx),
    'temperature':float(TEMPERATURE),
    'accept_threshold':float(ACCEPT_THRESHOLD),
    'target_accept_accuracy':float(TARGET_ACCEPTED_ACCURACY),
    'target_met_on_validation':bool(accept_result['meets_target_on_validation']),
    'vision_to_scene':NATURAL_SCENE_MAP,
    'version':'soundscape-cv-1.0',
}
torch.save(export_checkpoint,CHECKPOINT)
full_report={
    'release':'soundscape-cv-1.0',
    'training_timestamp_utc':datetime.now(timezone.utc).isoformat(),
    'dataset':DATASET_ID,'scope':label_names,'seed':SEED,'versions':VERSIONS,
    'data_audit':eda_report,
    'splits':{'train':len(train_idx),'validation':len(val_idx),'test':len(test_idx)},
    'hyperparameters':{'batch_size':BATCH_SIZE,'head_epochs_max':HEAD_EPOCHS,
                       'finetune_epochs_max':FINETUNE_EPOCHS,'patience':PATIENCE,
                       'finetune_feature_blocks':3},
    'validation':{'head_only':head_metrics,'partial_finetuning':fine_metrics,
                  'head_best_epoch':head_meta,'finetune_best_epoch':fine_meta},
    'selected_phase':selected_phase,'temperature':TEMPERATURE,
    'calibration_validation':temp_stats,'acceptance_validation':accept_result,
    'held_out_test':test_metrics,'held_out_test_selective':selective_test,
    'limitations':[
        'Only buildings, forest, glacier, mountain, sea and street are supervised',
        'No training/validation for desert, river, lake, waterfall, rain, or wind labels',
        'Single-label scene model does not describe all elements in complex images',
        'Confidence threshold and calibration do not guarantee out-of-distribution detection',
        'Visual appearance cannot prove audible sound / rain strength / relaxation',
        'Near duplicates are diagnostic only; cross-split perceptual duplicates may remain',
        'Dataset uploader license does not verify all original photo provenance',
        'Evaluation is from the source dataset; external phone-photo performance is unknown',
    ],
}
(OUT_DIR/'cv_evaluation.json').write_text(json.dumps(full_report,indent=2,ensure_ascii=False),encoding='utf8')
head_history.to_csv(OUT_DIR/'head_training.csv',index=False)
fine_history.to_csv(OUT_DIR/'finetune_training.csv',index=False)
print('Model checkpoint:', CHECKPOINT, CHECKPOINT.stat().st_size/1024**2,'MiB')
print('Report:',OUT_DIR/'cv_evaluation.json')

## 13. Production inference: user-supplied images

This function mirrors the platform's vision response and applies the exported calibration and reject policy. `scene=unknown` is expected for urban photos and when the model's reliability rule fails. User-correctable audio settings are separate from visual predictions.

In [ ]:
def predict_scene(pil_image, model=model, classes=label_names):
    image=ImageOps.exif_transpose(pil_image).convert('RGB')
    with torch.inference_mode():
        x=eval_transform(image).unsqueeze(0).to(DEVICE)
        logits=model(x).float().squeeze(0).cpu().numpy()
    probabilities=softmax(logits/TEMPERATURE)
    order=np.argsort(probabilities)[::-1]
    idx=int(order[0]);raw_label=classes[idx];confidence=float(probabilities[idx])
    accepted=bool(confidence>=ACCEPT_THRESHOLD)
    scene=NATURAL_SCENE_MAP.get(raw_label,'unknown') if accepted else 'unknown'
    return {'scene':scene,'raw_label':raw_label,'score':round(confidence,4),
            'accepted':accepted,'model':'mobilenet_finetuned' if selected_phase=='finetune' else 'mobilenet_head',
            'candidates':[{'label':classes[int(j)],'score':round(float(probabilities[j]),4)} for j in order[:4]],
            'caution':'Scene suggestion only. Scores are calibrated on in-domain validation, not proof of sound or OOD safety.'}

# Sanity-check the exact same example using the original image dataset.
example_id=int(test_idx[0]);sample_img=raw[example_id]['image']
print('True label:',label_names[int(labels_all[example_id])])
print(json.dumps(predict_scene(sample_img),indent=2))
display(sample_img)

## 14. Hands-on tests: upload your own photos

Use any real `jpg/png/webp` photos on Colab (forest, beach, desert, indoor, etc.) to inspect system behavior. This is **a product acceptance check**, not part of the held-out test score. You can also run it on a laptop by changing file paths. The notebook does **not** claim that an unsupported scene was trained or correctly recognized.

In [ ]:
# Optional manual test cell; never block Colab 'Run all' on a file picker.
RUN_INTERACTIVE_UPLOAD = False  # Change to True and re-run THIS cell to upload photos.
IMAGE_PATHS=[]
if RUN_INTERACTIVE_UPLOAD:
    try:
        from google.colab import files
        print('Select one or more images to test.')
        uploaded=files.upload()
        IMAGE_PATHS=list(uploaded.keys())
    except ImportError:
        print('Not in Colab: set IMAGE_PATHS to local image paths instead.')
else:
    print('Manual upload test skipped during Run all. Set RUN_INTERACTIVE_UPLOAD=True to test new photos.')

for img_path in IMAGE_PATHS:
    try:
        with Image.open(img_path) as im:
            outcome=predict_scene(im)
            print('\n',img_path,'\n',json.dumps(outcome,indent=2))
            display(ImageOps.exif_transpose(im).convert('RGB').resize((480,320)))
    except (OSError,ValueError) as exc:
        print('Cannot process',img_path,':',exc)

## 15. Vision → Audio integration contract and sample controls

The computer-vision model outputs **visual scene only**, not a full acoustic description. The user can correct its scene label and choose a sound family, acoustic character, and comfort mode. The audio team will use the resulting prompt/settings contract.

In [ ]:
SUPPORTED_APP_SCENES=['forest','coast','desert','river','waterfall','mountain','lake','unknown']
SUPPORTED_SOUNDS=['rain','sea_waves','wind','pouring_water','thunderstorm']
SCENE_PROMPTS={
    'forest':'a peaceful forest','coast':'a quiet ocean coastline','desert':'a wide desert landscape',
    'river':'a flowing river valley','waterfall':'a scenic waterfall','mountain':'a remote mountain landscape',
    'lake':'a calm lake shore','unknown':'an outdoor natural environment'}
SOUND_PROMPTS={
    'rain':'gentle natural rain','sea_waves':'ocean waves rolling onto the shore',
    'wind':'natural wind blowing','pouring_water':'flowing freshwater',
    'thunderstorm':'distant thunder and rainfall'}

def create_audio_request(vision_result,scene_override=None,sound='rain',intensity='gentle',comfort=True,seconds=10):
    scene=scene_override or vision_result.get('scene','unknown')
    assert scene in SUPPORTED_APP_SCENES and sound in SUPPORTED_SOUNDS
    assert intensity in ('gentle','moderate','strong') and 5<=int(seconds)<=30
    character={'gentle':'very soft and delicate','moderate':'moderately present','strong':'distinct and powerful'}[intensity]
    style='smooth relaxing ambience, without sudden loud events' if comfort else 'natural dynamic ambience'
    return {'scene':scene,'sound':sound,'intensity':intensity,'comfort_mode':bool(comfort),
            'seconds':int(seconds),
            'prompt':f'{SOUND_PROMPTS[sound]} in {SCENE_PROMPTS[scene]}, {character}, {style}, no speech or music.',
            'note':'Scene may be user-corrected. Audio character is requested, not detected in the photo.'}

vision_example=predict_scene(sample_img)
request=create_audio_request(vision_example,scene_override='forest',sound='rain',intensity='gentle')
print(json.dumps(request,indent=2))

## 16. Release the complete CV artifacts

The ZIP includes the **actual trained checkpoint produced by this run**, figures, train/validation/test indices, EDA audit, and final evaluation report. Download it now: Colab's temporary file system may be removed when the runtime disconnects.

For the platform: copy `scene_mobilenetv3.pt` to `models/scene_mobilenetv3.pt` (see project README). The server will use it automatically; **no re-training is required on the server**.

In [ ]:
assert CHECKPOINT.exists() and CHECKPOINT.stat().st_size>1_000_000
assert (OUT_DIR/'cv_evaluation.json').exists()
release_archive=Path(shutil.make_archive('SoundScape_CV_Final_Artifacts','zip',root_dir='.',base_dir=OUT_DIR.name))
print('RELEASE READY:',release_archive.resolve())
print('Includes:',sorted(p.name for p in OUT_DIR.iterdir()))
try:
    from google.colab import files
    files.download(str(release_archive))
except ImportError:
    print('Outside Colab: release ZIP is saved next to the notebook.')

## Final acceptance checklist

- [ ] Full dataset audited; splits saved; sample images and class distribution inspected.
- [ ] Head and partial fine-tuning completed; validation comparison recorded.
- [ ] Held-out test evaluation, confusion matrix, and error analysis generated.
- [ ] Calibration/abstention policy selected on **validation**, not on test.
- [ ] One or more **new** photographs inspected (especially an unsupported scene).
- [ ] `SoundScape_CV_Final_Artifacts.zip` downloaded and checkpoint copied into FastAPI project.
- [ ] In the platform: upload photo → analyze → correct scene if needed → compose prompt.
- [ ] Integrate the audio teammate's model and run image→prompt→generated WAV end-to-end tests before claiming full audio generation.

### Boundaries of this release

This is a **final, end-to-end training and export notebook**, not a toy quick-start. However, the checkpoint and evidence are generated when **you execute it**. It cannot replace the audio teammate's yet-to-be-delivered model, and it cannot independently validate unseen nature classes. For new classes (desert, waterfall, river, lake), acquire **properly labeled** photos, update the class taxonomy and retrain/re-evaluate as a separate release.